# Part.1

# DLT SQL構文の基本

このノートブックでは、Delta Live Tables（DLT）を使用して、クラウドオブジェクトストレージに着信するJSONファイルからの生データを処理し、データレイクハウスで分析ワークロードを実行する一連のテーブルを作成します。ここでは、データがパイプラインを介してフローする際に段階的に変換および拡張されるメダリオンアーキテクチャを示します。このノートブックでは、このアーキテクチャではなく、DLTのSQL構文に焦点を当てていますが、設計の概要も簡単に説明します。

* ブロンズテーブルにはJSONから読み込まれ、レコードがどのように投入されたかを説明するデータで拡張された生のレコードが含まれています。
* シルバーテーブルは、興味のあるフィールドの検証および拡張を行います。
* ゴールドテーブルには、ビジネスインサイトおよびダッシュボードに駆動するための集計データが含まれています。

## 学習目標

このノートブックの最後までに、学生は次の点について自信を持つはずです：
* Delta Live Tablesの宣言
* Auto Loaderを使用したデータの取り込み
* DLTパイプラインでのパラメータの使用
* 制約を使用したデータ品質の強制
* テーブルへのコメントの追加
* ライブテーブルとストリーミングライブテーブルの構文と実行の違いの説明


## DLTライブラリノートブックについて

DLTの構文は、ノートブックで対話的に実行するために意図されていません。このノートブックは、正確な実行を確認するためにはDLTパイプラインの一部としてスケジュールする必要があります。

DLTノートブックセルを対話的に実行する場合、文法的には正しいとのメッセージが表示されるはずです。このメッセージが返される前に一部の構文チェックが行われますが、クエリが望むように機能することを保証するものではありません。DLTコードの開発とトラブルシューティングについては、後のコースで詳しく説明します。

## パラメータ化

DLTパイプラインの構成中に、いくつかのオプションが指定されました。その中の1つが**Configurations**フィールドに追加されたキーと値のペアでした。
DLTパイプラインの構成でのConfigurationsは、Databricks JobsのパラメータやDatabricksノートブックのウィジェットに類似しています。
これらのレッスン全体で、SQLクエリ内で構成で設定されたファイルパスを含めるために **`${source}`** を使用します。

## クエリ結果としてのテーブル

Delta Live Tablesは、標準のSQLクエリを適応して、DDL（データ定義言語）とDML（データ操作言語）を統一された宣言的な構文に統合します。

DLTで作成できる2つの異なる種類の永続テーブルがあります：
- **Liveテーブル**は、レイクハウスのためのマテリアライズドビューであり、各リフレッシュでクエリの現在の結果を返します。
- **Streaming Liveテーブル**は、増分的でほぼリアルタイムのデータ処理を目的としています。

これらのオブジェクトは、Delta Lakeプロトコルで保存される永続テーブルとして取り扱われます（ACIDトランザクション、バージョニングなどの多くの利点を提供します）。LiveテーブルとStreaming Liveテーブルの違いについては、後のノートブックで詳しく説明します。

どちらのタイプのテーブルも、わずかに変更されたCTAS（create table as select）ステートメントのアプローチを採用しています。エンジニアはデータを変換するためのクエリを書くことだけに注意すれば、DLTが残りを処理します。

SQL DLTクエリの基本的な構文は次のとおりです：

**`CREATE OR REFRESH [STREAMING] LIVE TABLE table_name`**<br/>
**`AS select_statement`**<br/>


## Auto Loaderを使用したStreaming Ingestion

Databricksは、[Auto Loader](https://docs.databricks.com/ingestion/auto-loader/index.html)機能を開発して、クラウドオブジェクトストレージからDelta Lakeにデータを増分的にロードするための最適化された実行を提供しています。Auto LoaderをDLTと組み合わせるのは簡単です：ソースデータディレクトリを構成し、いくつかの設定を提供し、ソースデータに対してクエリを書きます。Auto Loaderは、新しいデータファイルがソースのクラウドオブジェクトストレージの場所に着信すると、それらを自動的に検出し、高価なスキャンや無限に増加するデータセットに対して結果を再計算する必要はありません。

**`cloud_files()`** メソッドを使用すると、Auto LoaderをSQLとネイティブに組み合わせることができます。このメソッドは、次の位置パラメーターを取ります：
* ソースの場所。これはクラウドベースのオブジェクトストレージである必要があります。
* ソースデータの形式。この場合はJSONです。
* オプションのリーダーオプションの任意のサイズのカンマ区切りのリスト。この場合、 **`cloudFiles.inferColumnTypes`** を **`true`** に設定します。

以下のクエリでは、ソースに含まれるフィールドに加えて、各レコードがいつインジェストされたかと、各レコードの特定のファイルソースに関する情報をキャプチャするために、Spark SQL関数 **`current_timestamp()`** および **`input_file_name()`** が使用されています。

In [0]:
%sql
CREATE OR REFRESH STREAMING LIVE TABLE orders_bronze
AS SELECT current_timestamp() processing_time, input_file_name() source_file, *
FROM cloud_files("csv/DLT", "csv", map("cloudFiles.inferColumnTypes", "true"))

## データの検証、拡張、変換

DLTを使用すると、標準的なSpark変換の結果からテーブルを簡単に宣言できます。DLTはデータセットのドキュメント化に他の場所で使用されているSpark SQLの機能を活用しながら、データ品質のチェックのための新しい機能を追加します。

以下のクエリの構文を分解してみましょう。

### Select文

select文にはクエリのコアロジックが含まれています。この例では、次のことを行っています。
* フィールド **`order_timestamp`** をタイムスタンプ型にキャストします
* すべての残りのフィールドを選択します（元の **`order_timestamp`** を含む、興味のない3つのフィールドを除く）

**FROM**句には2つの構造があるかもしれませんが、これにはなれていないかもしれません：
* **`LIVE`** キーワードは、スキーマ名の代わりに、現在のDLTパイプラインの構成されたターゲットスキーマを指すために使用されます
* **`STREAM`** メソッドは、SQLクエリのストリーミングデータソースを宣言するために使用されます

なお、パイプラインの構成時にターゲットスキーマが宣言されていない場合、テーブルは公開されません（つまり、メタストアに登録されず、他の場所でクエリできるようにはなりません）。ターゲットスキーマは、異なる実行環境間を移動する際に簡単に変更できます。これにより、同じコードがリージョナルなワークロードに対して簡単に展開されたり、開発環境から本番環境に昇格したりする際に、スキーマ名をハードコードする必要がありません。

### データ品質の制約

DLTは、データに品質の強制チェックを許可するためのシンプルなブール文を使用します。以下の文では、次のことを行います。
* **`valid_date`** という名前の制約を宣言します
* 条件付きチェックを定義します。 **`order_timestamp`** フィールドには、2021年1月1日よりも後の値が含まれている必要があります
* 任意のレコードが制約を犯す場合、DLTにはトランザクションを失敗させるように指示します

各制約には複数の条件がある可能性があり、1つのテーブルに対して複数の制約を設定できます。更新に失敗するだけでなく、制約の違反は、これらの無効なレコードを処理しながら、自動的にレコードを削除するか、違反の数だけを記録することもできます。

### テーブルコメント

テーブルコメントは標準的なSQLであり、組織全体で有用な情報を提供するために使用できます。この例では、テーブルに関するデータがどのようにインジェストされ、強制されているかについての簡単な人間が読める説明を書いています（これは他のテーブルメタデータを確認することでもわかるかもしれません）。

### テーブルプロパティ

**`TBLPROPERTIES`**  フィールドは、データのカスタムタグ付けのためのキー/バリューペアを任意の数だけ渡すために使用できます。ここでは、 **`quality`** のキーに **`silver`** の値を設定しています。

なお、このフィールドはカスタムタグを任意に設定できるようになっていますが、テーブルのパフォーマンスを制御するいくつかの設定を指定するためにも使用されます。テーブルの詳細を確認する際には、テーブルが作成されるたびにデフォルトでオンになるいくつかの設定にも遭遇するかもしれません。


In [0]:
%sql
CREATE OR REFRESH STREAMING LIVE TABLE orders_silver
(CONSTRAINT valid_date EXPECT (order_timestamp > "2021-01-01") ON VIOLATION FAIL UPDATE)
COMMENT "Append only orders with valid timestamps"
TBLPROPERTIES ("quality" = "silver")
AS SELECT timestamp(order_timestamp) AS order_timestamp, * EXCEPT (order_timestamp, source_file, _rescued_data)
FROM STREAM(LIVE.orders_bronze)

## Live Tables vs. Streaming Live Tables

これまでに確認した2つのクエリは、どちらもストリーミングライブテーブルを作成しています。以下では、いくつかの集計データのライブテーブル（またはマテリアライズドビュー）を返す単純なクエリを見てみましょう。

Sparkは歴史的に、バッチクエリとストリーミングクエリを区別してきました。ライブテーブルとストリーミングライブテーブルにも似たような違いがあります。

ライブテーブルとストリーミングライブテーブルの唯一の構文の違いは、作成句での **`STREAMING`** キーワードの欠如と、ソーステーブルを **`STREAM()`** メソッドでラップしないことです。

以下は、これらのテーブルの種類の違いのいくつかです。

### ライブテーブル
* 常に「正確」で、更新後にその内容が定義と一致します。
* すべてのデータでまるでテーブルが初めて定義されたかのような結果を返します。
* DLTパイプライン外の操作によって変更されるべきではありません（未定義の回答が得られるか、変更が元に戻されるかもしれません）。

### ストリーミングライブテーブル
* 「追加のみ」のストリーミングソースからの読み取りのみサポートします。
* 各入力バッチは一度だけ読み取り、それがどのように変更されても（結合されたディメンションが変更されても、クエリの定義が変更されてもなど）、一度だけです。
* 管理対象のDLTパイプラインの外でテーブル上で操作を実行できます（データの追加、GDPRの実行など）。


In [0]:
%sql
CREATE OR REFRESH LIVE TABLE orders_by_date
AS SELECT date(order_timestamp) AS order_date, count(*) AS total_daily_orders
FROM LIVE.orders_silver
GROUP BY date(order_timestamp)

## 要約

このノートブックを確認することで、次のことができるようになるはずです。
* Delta Live Tablesの宣言
* Auto Loaderを使用したデータの読み込み
* DLTパイプラインでのパラメータの使用
* 制約を使用したデータ品質の強制
* テーブルへのコメントの追加
* ライブテーブルとストリーミングライブテーブルの構文と実行の違いの説明

次のノートブックでは、これらの構文的な構造を学びながら、新しい概念もいくつか追加していきます。

# Part.2

# DLT SQL構文のさらなる理解

DLTパイプラインを使用すると、1つまたは複数のノートブックを使用して、複数のデータセットを単一のスケーラブルなワークロードに組み合わせることが簡単になります。

前回のノートブックでは、クラウドオブジェクトストレージからのJSONデータの取り込みを含む一連のクエリを使用して、DLT構文の基本的な機能のいくつかを確認しました。このノートブックも同様にメダリオンアーキテクチャに従っており、いくつかの新しい概念を紹介しています。
* 生のレコードは、顧客に関する変更データの取り込み（CDC）情報を表します
* ブロンズテーブルは、再びAuto Loaderを使用してクラウドオブジェクトストレージからJSONデータを取り込みます
* 制約を強制するためのテーブルが定義され、レコードがシルバーレイヤに渡される前に制約が適用されます
* **`APPLY CHANGES INTO`** を使用してCDCデータを自動的にシルバーレイヤに処理し、Type 1 <a href="https://en.wikipedia.org/wiki/Slowly_changing_dimension" target="_blank">スローリーチェンジングディメンション（SCD）テーブル</a>にします
* ゴールドテーブルが定義され、このType 1テーブルの現在のバージョンから集計を計算します
* 別のノートブックで定義されたテーブルと結合するビューが定義されています

## 学習目標

このレッスンの最後までに、学生は次のことに慣れているはずです。
* **`APPLY CHANGES INTO`** を使用してCDCデータを処理する
* ライブビューの宣言
* ライブテーブルの結合
* DLTライブラリノートブックがパイプラインで協力する方法の説明
* DLTパイプラインで複数のノートブックをスケジュールする


## Auto Loaderを使用してデータを取り込む

前回のノートブックと同様に、Auto Loaderで構成されたデータソースに対してブロンズテーブルを定義します。

以下のコードでは、スキーマを推論せずにJSONからデータを取り込むためのAuto Loaderオプションが省略されています。スキーマが提供されず、または推論されない場合、フィールドは正しい名前でありながらすべて**`STRING`**型として保存されます。

以下のコードでは、シンプルなコメントが提供され、データ取り込みのタイミングと各レコードのファイル名が追加されています。

In [0]:
%sql
CREATE OR REFRESH STREAMING LIVE TABLE customers_bronze
COMMENT "Raw data from customers CDC feed"
AS SELECT current_timestamp() processing_time, input_file_name() source_file, *
FROM cloud_files("${source}/customers", "json")


## データ品質の強化の続き

以下のクエリでは、以下のポイントが示されています。
- 制約が違反した場合の3つの動作オプション
- 複数の制約を持つクエリ
- 同じ制約に複数の条件を提供
- 制約内で組み込みのSQL関数を使用

データソースについて：
- データは **`INSERT`**、**`UPDATE`**、**`DELETE`** 操作を含むCDCフィードです。
- 更新と挿入の操作はすべてのフィールドに有効なエントリを含む必要があります。
- 削除操作は、タイムスタンプ、 **`customer_id`** 、および操作フィールド以外のすべてのフィールドに **`NULL`** 値を含める必要があります。

私たちのシルバーテーブルに良質なデータだけが入るようにするために、削除操作の予想される **`NULL`** 値を無視する品質強化ルールの一連を作成します。

以下では、これらの各制約を詳細に説明します。

**`valid_id`**
この制約は、 **`customer_id`** フィールドに **`NULL`** 値が含まれている場合、トランザクションを失敗させます。

**`valid_operation`**
この制約は、 **`operation`** フィールドに **`NULL`** 値が含まれているレコードを削除します。

**`valid_address`**
この制約は、 **`operation`** フィールドが **`DELETE`** でない場合、住所を構成する4つのフィールドのいずれかに **`NULL`** 値が含まれているかどうかを確認します。制約内の無効なレコードに対する追加の指示がないため、違反した行はメトリクスに記録されますが、削除されません。

**`valid_email`**
この制約は、 **`email`** フィールドの値が有効なメールアドレスであるかどうかを確認するために正規表現のパターンマッチングを使用します。 **`operation`** フィールドが **`DELETE`** である場合はこれを適用しないようにロジックが組まれています（なぜならこれらは **`email`** フィールドに **`NULL`** 値を持つことになるからです）。違反したレコードは削除されます。


In [0]:
%sql
CREATE STREAMING LIVE TABLE customers_bronze_clean
(CONSTRAINT valid_id EXPECT (customer_id IS NOT NULL) ON VIOLATION FAIL UPDATE,
CONSTRAINT valid_operation EXPECT (operation IS NOT NULL) ON VIOLATION DROP ROW,
CONSTRAINT valid_name EXPECT (name IS NOT NULL or operation = "DELETE"),
CONSTRAINT valid_address EXPECT (
  (address IS NOT NULL and 
  city IS NOT NULL and 
  state IS NOT NULL and 
  zip_code IS NOT NULL) or
  operation = "DELETE"),
CONSTRAINT valid_email EXPECT (
  rlike(email, '^([a-zA-Z0-9_\\-\\.]+)@([a-zA-Z0-9_\\-\\.]+)\\.([a-zA-Z]{2,5})$') or 
  operation = "DELETE") ON VIOLATION DROP ROW)
AS SELECT *
  FROM STREAM(LIVE.customers_bronze)

上記のクエリでは、CDCデータを処理するための新しい構文構造である **`APPLY CHANGES INTO`** が紹介されています。

**`APPLY CHANGES INTO`** は、以下の保証と要件があります。
* CDCデータの増分/ストリーミング取り込みを実行します。
* 1つまたは複数のフィールドをテーブルの主キーとして指定するためのシンプルな構文を提供します。
* デフォルトの仮定は、行には挿入と更新が含まれているということです。
* オプションで削除を適用できます。
* ユーザーが提供したシーケンスキーを使用して、遅延して到着するレコードを自動的に並べ替えます。
* **`EXCEPT`** キーワードを使用して無視する列を指定するためのシンプルな構文を使用します。
* 変更をType 1 SCDとして適用するというデフォルトがあります。

上記のコードは、次のことを行っています。
* **`customers_silver`** テーブルを作成します。 **`APPLY CHANGES INTO`** は、変更が適用される対象のテーブルを別のステートメントで宣言する必要があります。
* **`customers_silver`** テーブルを変更が適用される対象として識別します。
* ストリーミングソースとして **`customers_bronze_clean`** テーブルを指定します。
* **`customer_id`** を主キーとして指定します。
* **`operation`** フィールドが **`DELETE`** の場合は削除として適用されるレコードを指定します。
* オペレーションが適用される方法を指定するために **`timestamp`** フィールドを指定します。
* **`operation`** 、 **`source_file`** 、および **`_rescued_data`** 以外のすべてのフィールドを対象テーブルに追加するように指定します。

In [0]:
%sql
CREATE OR REFRESH STREAMING LIVE TABLE customers_silver;

APPLY CHANGES INTO LIVE.customers_silver
  FROM STREAM(LIVE.customers_bronze_clean)
  KEYS (customer_id)
  APPLY AS DELETE WHEN operation = "DELETE"
  SEQUENCE BY timestamp
  COLUMNS * EXCEPT (operation, source_file, _rescued_data)


**`APPLY CHANGES INTO`** は、デフォルトでType 1 SCDテーブルを作成するようになっており、つまり、各ユニークキーには最大1つのレコードしかなく、更新は元の情報を上書きします。

前のセルでの操作の対象がストリーミングライブテーブルとして定義されていましたが、このテーブルではデータが更新および削除されています（したがって、ストリーミングライブテーブルソースの追加専用要件が壊れています）。 したがって、ダウンストリームの操作はこのテーブルに対してストリーミングクエリを実行することはできません。

このパターンにより、更新が順番外れで到着した場合、ダウンストリームの結果を正しく再計算して更新を反映できるようになります。 また、ソーステーブルからレコードが削除された場合、これらの値はパイプライン内の後のテーブルにはもはや反映されないようになります。

以下では、 **`customers_silver`** テーブルのデータからライブテーブルを作成するための簡単な集計クエリを定義しています。


In [0]:
%sql
CREATE LIVE TABLE customer_counts_state
  COMMENT "Total active customers per state"
AS SELECT state, count(*) as customer_count, current_timestamp() updated_at
  FROM LIVE.customers_silver
  GROUP BY state


以下のクエリでは、 **`VIEW`** キーワードで **`TABLE`** を置き換えることで、DLTビューを定義しています。

DLTのビューは永続テーブルとは異なり、オプションで **`STREAMING`** として定義することができます。

ビューはライブテーブルと同じ更新の保証を持っていますが、クエリの結果はディスクに保存されません。

Databricksの他の場所で使用されるビューとは異なり、DLTビューはメタストアに永続されないため、それらは所属するDLTパイプライン内からのみ参照できます（これはほとんどのSQLシステムの一時ビューと同じスコーピングです）。

ビューは引き続きデータ品質を強制するために使用でき、ビューのメトリクスはテーブルのように収集および報告されます。

## Joins and Referencing Tables Across Notebook Libraries

これまでに確認したコードは、2つのソースデータセットが異なるノートブックで一連の手順を伝播するのを示しています。

DLTは、1つのDLTパイプライン構成の一部として複数のノートブックをスケジュールすることをサポートしています。既存のDLTパイプラインを編集して追加のノートブックを追加できます。

DLTパイプライン内では、任意のノートブックライブラリで作成されたテーブルとビューを参照できます。

基本的には、 **`LIVE`** キーワードによって参照されるスキーマのスコープを個々のノートブックではなくDLTパイプラインレベルと考えることができます。

以下のクエリでは、 **`orders`** および **`customers`** データセットからのシルバーテーブルを結合して新しいビューを作成しています。このビューはストリーミングとして定義されていないため、常に各顧客の現在の有効な **`email`** がキャプチャされ、 **`customers_silver`** テーブルから削除された後はレコードが自動的に削除されます。


In [0]:
%sql
CREATE LIVE VIEW subscribed_order_emails_v
  AS SELECT a.customer_id, a.order_id, b.email 
    FROM LIVE.orders_silver a
    INNER JOIN LIVE.customers_silver b
    ON a.customer_id = b.customer_id
    WHERE notifications = 'Y'


## このノートブックをDLTパイプラインに追加

既存のパイプラインに追加のノートブック ライブラリを追加するのは、DLT UI を使用して簡単に行えます。

1. このコースで既に構成したDLTパイプラインに移動します
1. 画面右上の **Settings** ボタンをクリックします
1. **Notebook Libraries** の下で、**Add notebook library** をクリックします
   * ファイルピッカーを使用してこのノートブックを選択し、**Select** をクリックします
1. 更新を保存するために **Save** ボタンをクリックします
1. 画面右上の青い **Start** ボタンをクリックして、パイプラインを更新し、新しいレコードを処理します

<img src="https://files.training.databricks.com/images/icon_hint_24.png"> このノートブックへのリンクは、[DE 4.1 - DLT UI Walkthrough]($../DE 4.1 - DLT UI Walkthrough)の<br/>
**Task #2** の **Generate Pipeline Configuration** セクションの指示の中にあります



## 概要

このノートブックを確認することで、以下の作業に慣れるようになるでしょう：
* **`APPLY CHANGES INTO`** を使用して CDC データを処理する
* ライブビューを宣言する
* ライブテーブルを結合する
* DLT ライブラリ ノートブックがどのように連携するかを説明する
* DLT パイプラインで複数のノートブックをスケジュールする

次のノートブックでは、パイプラインの出力を探索します。その後、DLT コードの反復的な開発とトラブルシューティングの方法を見ていきます。
